In [1]:
import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import lm_eval
from lm_eval.tasks import TaskManager
from lm_eval.evaluator import simple_evaluate
from lm_eval.utils import make_table

# Constants

In [3]:
TAWJEEH_DATASET_NAME = 'ArSarcasm_v2'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/ArSarcasm_v2_experimental'
TASK_NAME='sarcasm_detection'
MODEL_PATH = "/raid_storage/shared_models/Qwen3-8B"
MODEL_NAME = "Qwen3-8B-chat"
TUNED_MODEL_PATH = None
BATCH_SIZE = 40

In [4]:
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [5]:
import requests
 
from tqdm.auto import tqdm
 
prompts = None
 
tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts:
    raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14901,
  'tags': [],
  'name': 'A Simple Test Prompt',
  'task': {'name': 'dialect identification'},
  'status': 'DRAFT',
  'template': 'Please predict the most suitable dialect for the following text: {{arabic}}\xa0\r\n|||{{answer_choices[label]}}',
  'created_by': 'irfan',
  'dataset_name': 'arbml/AraBench_dev',
  'dataset_subset': 'default',
  'answer_choices': ['Tunisian',
   'MSA',
   'Morrocan',
   'Qatari',
   'Egyptian',
   'Lebanese'],
  'text_direction': 'ltr'},
 {'id': 14898,
  'tags': ['', 'Zero-shot COT'],
  'name': 'Prompt with zero-shot chain of thoughts',
  'task': {'name': 'claim verification'},
  'status': 'APPROVED',
  'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
  'created_by': 'ahmed',


In [6]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

352

### Get the dataset prompts

In [7]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

8

In [8]:
SELECTED_PROMPTS_IDS = [
    14779,
    14802,
    14835,
    14837,
    14838,
]

In [9]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [10]:
import datasets

In [11]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['tweet', 'sentiment', 'dialect', 'label'],
        num_rows: 12548
    })
    test: Dataset({
        features: ['tweet', 'sentiment', 'dialect', 'label'],
        num_rows: 3000
    })
})

### Merge the prompts

In [12]:
from jinja2 import Environment, StrictUndefined

In [13]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        sample['answer_choices'] = prompt_template['answer_choices']
        env = Environment(undefined=StrictUndefined)
        if "|||" not in template:
            raise ValueError("No ||| dividor")
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e

Perform generation on one example prompt, for experimentation

In [14]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['test'][1]))

Task: Identify whether the following tweet is sarcastic or non-sarcastic by comparing it to typical sarcasm cues.

Indicators of Sarcasm:
- Exaggerated praise or criticism
- Statements that imply the opposite of their literal meaning
- Use of irony or unexpected humor

Tweet: الخبل ما عرف يتحرش الا في الكويتين غاسل بالفلوس تركي بعر الطيور علي اشكالها تقع اليوم يا ولييدوه بتصير بالامارتي نسيحه  

Answer: Based on the indicators, respond with "sarcastic" if the tweet shows sarcasm, or "non-sarcastic" if it does not.
|||
Non sarcastic


In [15]:
for prompt in dataset_prompts:
    prompt['merged_samples'] = list(
        map(
            lambda sample: apply_template(prompt, sample),
            tqdm(hf_exp_dataset['test']),
        )
    )
    prompt['original_samples'] = list(hf_exp_dataset['test'])

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

# Evaluate on each prompt and report the results

In [16]:
from datasets import DatasetDict

def create_hf_dataset(dataset_prompt, columns=None):
  if columns is None:
    columns = ['text', 'label','choices']
  texts = []
  labels = []
  choices = []
  for i,merged_sample in enumerate(dataset_prompt['merged_samples']):
    prefix= merged_sample.split('|||')[0]
    prefix = prefix.strip()
    output = merged_sample.split('|||')[1].strip()
    example_choices = dataset_prompt['answer_choices']
    texts.append(prefix)
    labels.append(output)
    choices.append(example_choices)
  dataset = DatasetDict({ 'test' : datasets.Dataset.from_dict({
      columns[0]: texts,
      columns[1]: labels,
      columns[2]: choices,
  })})
  return dataset

In [17]:
dataset = create_hf_dataset(dataset_prompts[0])
dataset['test'][1]['text']

'Task: Identify whether the following tweet is sarcastic or non-sarcastic by comparing it to typical sarcasm cues.\n\nIndicators of Sarcasm:\n- Exaggerated praise or criticism\n- Statements that imply the opposite of their literal meaning\n- Use of irony or unexpected humor\n\nTweet: الخبل ما عرف يتحرش الا في الكويتين غاسل بالفلوس تركي بعر الطيور علي اشكالها تقع اليوم يا ولييدوه بتصير بالامارتي نسيحه  \n\nAnswer: Based on the indicators, respond with "sarcastic" if the tweet shows sarcasm, or "non-sarcastic" if it does not.'

In [18]:
import torch
from lm_eval.models.vllm_causallms import VLLM
kwargs = dict(
    pretrained=MODEL_PATH,
    trust_remote_code=True,
    tensor_parallel_size=torch.cuda.device_count(),
    tokenizer=TOKENIZER_PATH,
    gpu_memory_utilization=0.9,
)

lm_obj = VLLM(**kwargs)

INFO 03-08 12:09:53 [utils.py:223] non-default args: {'tokenizer': '/raid_storage/shared_models/Qwen3-8B', 'trust_remote_code': True, 'seed': 1234, 'disable_log_stats': True, 'model': '/raid_storage/shared_models/Qwen3-8B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-08 12:09:53 [model.py:529] Resolved architecture: Qwen3ForCausalLM
INFO 03-08 12:09:53 [model.py:1549] Using max model len 40960
INFO 03-08 12:09:53 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-08 12:09:53 [vllm.py:689] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=1056593) INFO 03-08 12:09:53 [core.py:97] Initializing a V1 LLM engine (v0.16.0) with config: model='/raid_storage/shared_models/Qwen3-8B', speculative_config=None, tokenizer='/raid_storage/shared_models/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsCon

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:06 [default_loader.py:293] Loading weights took 4.28 seconds
(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:07 [gpu_model_runner.py:4221] Model loading took 15.27 GiB memory and 5.017112 seconds
(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:15 [backends.py:916] Using cache directory: /raid_storage/SLURM/home/slurm_majedalshaibani/.cache/vllm/torch_compile_cache/cb30c65974/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:15 [backends.py:976] Dynamo bytecode transform time: 8.30 s
(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:22 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.347 s
(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:22 [monitor.py:34] torch.compile takes 10.64 s in total
(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:24 [gpu_worker.py:373] Available KV cache memory: 54.57 GiB
(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:24 [kv_cache_util

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:02<00:00, 20.90it/s]
Capturing CUDA graphs (decode, FULL): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 35/35 [00:01<00:00, 23.87it/s]


(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:28 [gpu_model_runner.py:5246] Graph capturing finished in 5 secs, took 0.60 GiB
(EngineCore_DP0 pid=1056593) INFO 03-08 12:10:28 [core.py:278] init engine (profile, create kv cache, warmup model) took 21.81 seconds
INFO 03-08 12:10:30 [llm.py:355] Supported tasks: ['generate']


In [19]:
def evaluate_tasks(tasks,dataset_sub_path=TAWJEEH_DATASET_NAME):
    # MAKE SURE THE NOTEBOOK IS RUNNING FROM THE PROJECT ROOT!
    task_manager = TaskManager(include_path=f"eval_harness_extra_tasks/{dataset_sub_path}")
    results = simple_evaluate(  # call simple_evaluate
        model=lm_obj,
        tasks=tasks,
        num_fewshot=0,
        task_manager=task_manager,
    )
    return results

In [20]:
import json

def create_and_evaluate_single_prompt(prompt, save_results=True, force_re_evaluate=False):
    prompt_id = prompt['id']
    if TUNED_MODEL_PATH:
        results_dir = f'evaluation_results/{MODEL_NAME}-tuned/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    else:
        results_dir = f'evaluation_results/{MODEL_NAME}/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    prompt_results_file_path = f'{results_dir}/prompt_{prompt_id}.json'
    
    # Check if results exist and handle based on parameters
    if os.path.exists(prompt_results_file_path) and os.path.getsize(prompt_results_file_path) > 0:
        if not force_re_evaluate:
            print(f"Skipping prompt {prompt_id} - results already exist")
            with open(prompt_results_file_path, 'r') as f:
                prompt_results = json.load(f)
                print(make_table(prompt_results))
                return prompt_results
        else:
            print(f"Force re-evaluate enabled - reevaluating prompt {prompt_id}")
    
    # Create dataset and task files
    dataset = create_hf_dataset(prompt)
    
    # Save dataset
    dataset_dir = f'experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}'
    os.makedirs(dataset_dir, exist_ok=True)
    dataset['test'].to_parquet(f"{dataset_dir}/data.parquet")
    
    # Create YAML configuration
    yaml_text = f'''task: {TAWJEEH_DATASET_NAME}_prompt_{prompt_id}
dataset_path: experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}
output_type: multiple_choice
test_split: train
doc_to_text: text
doc_to_choice: choices
doc_to_target: label
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: True
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0'''
    
    # Save YAML
    yaml_dir = f'eval_harness_extra_tasks/{TAWJEEH_DATASET_NAME}'
    os.makedirs(yaml_dir, exist_ok=True)
    with open(f'{yaml_dir}/prompt_{prompt_id}.yaml', 'w') as f:
        f.write(yaml_text)
    
    # Evaluate single prompt
    evaluation_task_name = f'{TAWJEEH_DATASET_NAME}_prompt_{prompt_id}'
    prompt_results = evaluate_tasks(tasks=[evaluation_task_name])
    
    print(make_table(prompt_results))
    
    # Save results if save_results is True
    if save_results:
        os.makedirs(results_dir, exist_ok=True)
        with open(prompt_results_file_path, 'w') as f:
            json.dump(prompt_results, f, ensure_ascii=False, indent=4, 
                     default=lambda o: '<not serializable>')
        print(f"Saved results for prompt {prompt_id}")
    else:
        print(f"Results not saved for prompt {prompt_id} (save_results=False)")
    
    print(f"Completed evaluation for prompt {prompt_id}")
    return prompt_results

In [21]:
def evaluate_all_prompts_sequentially(dataset_prompts, **kwargs):
    print(f"Starting sequential evaluation of {len(dataset_prompts)} prompts")
    all_results = {}
    
    for i, prompt in enumerate(dataset_prompts, 1):
        print('-' * 80)
        print(f"\nProcessing prompt {i}/{len(dataset_prompts)} (ID: {prompt['id']})")
        print("Template:", prompt['template'])
        print('-' * 80)
        
        prompt_results = create_and_evaluate_single_prompt(prompt, **kwargs)
        all_results[f"{TAWJEEH_DATASET_NAME}_prompt_{prompt['id']}"] = prompt_results
    
    return {'results': all_results}

In [22]:
all_results = evaluate_all_prompts_sequentially(dataset_prompts=dataset_prompts)

Starting sequential evaluation of 5 prompts
--------------------------------------------------------------------------------

Processing prompt 1/5 (ID: 14838)
Template: Task: Identify whether the following tweet is sarcastic or non-sarcastic by comparing it to typical sarcasm cues.

Indicators of Sarcasm:
- Exaggerated praise or criticism
- Statements that imply the opposite of their literal meaning
- Use of irony or unexpected humor

Tweet: {{ tweet }}

Answer: Based on the indicators, respond with "sarcastic" if the tweet shows sarcasm, or "non-sarcastic" if it does not.
|||
{{ answer_choices[label] }}
--------------------------------------------------------------------------------


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

[2026-03-08 12:10:34] INFO evaluator.py:211: Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
[2026-03-08 12:10:34] INFO evaluator.py:267: Using pre-initialized model


Generating train split: 0 examples [00:00, ? examples/s]

[2026-03-08 12:10:35] INFO __init__.py:700: Selected tasks:
[2026-03-08 12:10:35] INFO __init__.py:691: Task: ArSarcasm_v2_prompt_14838 (eval_harness_extra_tasks/ArSarcasm_v2/prompt_14838.yaml)
[2026-03-08 12:10:35] WARNING evaluator.py:333: Overwriting default num_fewshot of ArSarcasm_v2_prompt_14838 from None to 0
[2026-03-08 12:10:35] INFO task.py:311: Building contexts for ArSarcasm_v2_prompt_14838 on rank 0...
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3000/3000 [00:00<00:00, 115036.40it/s]
[2026-03-08 12:10:35] INFO evaluator.py:584: Running loglikelihood requests
Running loglikelihood requests:   0%|                                                                                                                                                                                      | 0/6000 [00:00<?, ?it/s]

...

[2026-03-08 12:15:21] INFO __init__.py:700: Selected tasks:
[2026-03-08 12:15:21] INFO __init__.py:691: Task: ArSarcasm_v2_prompt_14779 (eval_harness_extra_tasks/ArSarcasm_v2/prompt_14779.yaml)
[2026-03-08 12:15:21] WARNING evaluator.py:333: Overwriting default num_fewshot of ArSarcasm_v2_prompt_14779 from None to 0
[2026-03-08 12:15:21] INFO task.py:311: Building contexts for ArSarcasm_v2_prompt_14779 on rank 0...
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3000/3000 [00:00<00:00, 106594.20it/s]
[2026-03-08 12:15:21] INFO evaluator.py:584: Running loglikelihood requests
Running loglikelihood requests:   0%|                                                                                                                                                                                      | 0/6000 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/6000 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …

Running loglikelihood requests: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6000/6000 [00:45<00:00, 131.55it/s]


|          Tasks          |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|-------------------------|------:|------|-----:|--------|---|-----:|---|-----:|
|ArSarcasm_v2_prompt_14779|      1|none  |     0|acc     |↑  |0.4293|±  |0.0090|
|                         |       |none  |     0|acc_norm|↑  |0.7253|±  |0.0082|

Saved results for prompt 14779
Completed evaluation for prompt 14779


In [ ]:
exit()

ERROR 03-08 12:16:15 [core_client.py:616] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.


: 